In [1]:
import pandas as pd
import numpy as np
from transformers import AutoTokenizer
import torch

In [2]:
import sys
sys.path.append("./cluster_anlys")

from pca_hdbscan import run_random_pca_hdbscan_grid

seed_list = [0, 42, 1000, 9999]

In [3]:
import random

base_seed = 0
rng = random.Random(base_seed)
seed_list = rng.sample(range(2**31 - 1), 4)
seed_list


[1813382118, 827307999, 1627694678, 1911784257]

# Mistral-7B

In [4]:
model_name = "mistralai/Mistral-7B-v0.1"
# ===== Step 1: Load tokenizer =====
# tokenizer_path = f"{model_name}/tokenizer" 
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
# print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

# ===== Step 2: Load lm_head.weight =====
space_name = "output_proj"
lm_head_path = f"{model_name}/tensors/{space_name}.pt" 
embedding_matrix = torch.load(lm_head_path, map_location="cpu")
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")
embedding_matrix = embedding_matrix.to(torch.float32)
# embedding_matrix = embedding_matrix.cpu().to(torch.float32).numpy()
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")

[✓] lm_head.weight loaded: shape = torch.Size([32000, 4096])
[✓] lm_head.weight loaded: shape = torch.Size([32000, 4096])


In [5]:
# =========================
# Config
# =========================

out_root = "comp"

full_pca_dim = 4096

candidate_dims = [5, 142, 997, 2084, 3031, 3459, 3938, 4088, 4096]

hdbscan_param_grid = [
    {
        "min_cluster_size": 5,
        "min_samples": 5,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "cluster_selection_epsilon": 0.0,
    },
    # if needed
    # {
    #     "min_cluster_size": 20,
    #     "min_samples": 10,
    #     "metric": "euclidean",
    #     "cluster_selection_method": "eom",
    #     "cluster_selection_epsilon": 0.0,
    # },
]

l2_norm = True
summary_filename = f"summary.csv"

In [6]:
all_summary = []

for seed in seed_list:
    df = run_random_pca_hdbscan_grid(
        embedding_matrix=embedding_matrix,
        candidate_dims=candidate_dims,
        hdbscan_param_grid=hdbscan_param_grid,
        model_name=model_name,
        space_name=space_name,
        pca_seed=seed,
        randomized_fit_dim=full_pca_dim,
        out_root="comp",
        l2_norm=True,
    )
    all_summary.append(df)

all_summary_df = pd.concat(all_summary, ignore_index=True)
all_summary_df.head()

[warn] randomized_fit_dim is ignored in the corrected per-dim randomized PCA pipeline.
[✓] Randomized PCA finished | model=mistralai/Mistral-7B-v0.1 dim=5 seed=1813382118 time=0.87s cum_var=0.031960
[✓] Saved to: comp/mistralai/Mistral-7B-v0.1/random_pca/seed_1813382118/pca_dim_5
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mistral-7B-v0.1 dim=142 seed=1813382118 time=2.18s cum_var=0.123498
[✓] Saved to: comp/mistralai/Mistral-7B-v0.1/random_pca/seed_1813382118/pca_dim_142
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mistral-7B-v0.1 dim=997 seed=1813382118 time=3.65s cum_var=0.437701
[✓] Saved to: comp/mistralai/Mistral-7B-v0.1/random_pca/seed_1813382118/pca_dim_997
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mistral-7B-v0.1 dim=2084 seed=1813382118 time=8.11s cum_var=0.714780
[✓] Saved to: comp/mistralai/Mistral-7B-v0.1/random_pca/seed_1813382118/pca_dim_2084
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mistral-7B-v0

,run_id,model_name,space_name,pca_mode,pca_seed,pca_source_dim,pca_dim,l2_norm,min_cluster_size,min_samples,...,cluster_selection_method,cluster_selection_epsilon,n_clusters_excl_noise,n_noise,n_total,noise_ratio,avg_prob_all,avg_prob_assigned,cluster_csv_path,meta_json_path
0,1,mistralai/Mistral-7B-v0.1,output_proj,randomized,1813382118,5,5,True,5,5,...,eom,0.0,322,27976,32000,0.874250,0.118114,0.939276,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...
1,2,mistralai/Mistral-7B-v0.1,output_proj,randomized,1813382118,142,142,True,5,5,...,eom,0.0,65,28832,32000,0.901000,0.085616,0.864808,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...
2,3,mistralai/Mistral-7B-v0.1,output_proj,randomized,1813382118,997,997,True,5,5,...,eom,0.0,638,24755,32000,0.773594,0.209079,0.923467,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...
3,4,mistralai/Mistral-7B-v0.1,output_proj,randomized,1813382118,2084,2084,True,5,5,...,eom,0.0,886,22060,32000,0.689375,0.292843,0.942756,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...
4,5,mistralai/Mistral-7B-v0.1,output_proj,randomized,1813382118,3031,3031,True,5,5,...,eom,0.0,933,21756,32000,0.679875,0.303543,0.948201,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...,comp/mistralai/Mistral-7B-v0.1/output_proj/ran...


# Mixtral-8x7B

In [7]:
from transformers import AutoTokenizer
import torch

model_name = "mistralai/Mixtral-8x7B-v0.1"
# ===== Step 1: Load tokenizer =====
# tokenizer_path = f"{model_name}/tokenizer" 
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
# print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

# ===== Step 2: Load lm_head.weight =====
space_name = "output_proj"
lm_head_path = f"{model_name}/tensors/{space_name}.pt" 
embedding_matrix = torch.load(lm_head_path, map_location="cpu")
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")
embedding_matrix = embedding_matrix.to(torch.float32)
# embedding_matrix = embedding_matrix.cpu().to(torch.float32).numpy()
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")

[✓] lm_head.weight loaded: shape = torch.Size([32000, 4096])
[✓] lm_head.weight loaded: shape = torch.Size([32000, 4096])


In [8]:
# =========================
# Config
# =========================

out_root = "comp"

full_pca_dim = 4096

candidate_dims = [8, 158, 1111, 2156, 3052, 3457, 3957, 4093, 4096]

hdbscan_param_grid = [
    {
        "min_cluster_size": 5,
        "min_samples": 5,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "cluster_selection_epsilon": 0.0,
    },
    # if needed
    # {
    #     "min_cluster_size": 20,
    #     "min_samples": 10,
    #     "metric": "euclidean",
    #     "cluster_selection_method": "eom",
    #     "cluster_selection_epsilon": 0.0,
    # },
]

l2_norm = True
summary_filename = f"summary.csv"

In [9]:
all_summary = []

for seed in seed_list:
    df = run_random_pca_hdbscan_grid(
        embedding_matrix=embedding_matrix,
        candidate_dims=candidate_dims,
        hdbscan_param_grid=hdbscan_param_grid,
        model_name=model_name,
        space_name=space_name,
        pca_seed=seed,
        randomized_fit_dim=full_pca_dim,
        out_root="comp",
        l2_norm=True,
    )
    all_summary.append(df)

all_summary_df = pd.concat(all_summary, ignore_index=True)
all_summary_df.head()

[warn] randomized_fit_dim is ignored in the corrected per-dim randomized PCA pipeline.
[✓] Randomized PCA finished | model=mistralai/Mixtral-8x7B-v0.1 dim=8 seed=1813382118 time=0.91s cum_var=0.030026
[✓] Saved to: comp/mistralai/Mixtral-8x7B-v0.1/random_pca/seed_1813382118/pca_dim_8
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mixtral-8x7B-v0.1 dim=158 seed=1813382118 time=2.04s cum_var=0.118824
[✓] Saved to: comp/mistralai/Mixtral-8x7B-v0.1/random_pca/seed_1813382118/pca_dim_158
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mixtral-8x7B-v0.1 dim=1111 seed=1813382118 time=3.94s cum_var=0.462395
[✓] Saved to: comp/mistralai/Mixtral-8x7B-v0.1/random_pca/seed_1813382118/pca_dim_1111
Memory Cleaned.
[✓] Randomized PCA finished | model=mistralai/Mixtral-8x7B-v0.1 dim=2156 seed=1813382118 time=7.85s cum_var=0.727686
[✓] Saved to: comp/mistralai/Mixtral-8x7B-v0.1/random_pca/seed_1813382118/pca_dim_2156
Memory Cleaned.
[✓] Randomized PCA finished | model=mistr

,run_id,model_name,space_name,pca_mode,pca_seed,pca_source_dim,pca_dim,l2_norm,min_cluster_size,min_samples,...,cluster_selection_method,cluster_selection_epsilon,n_clusters_excl_noise,n_noise,n_total,noise_ratio,avg_prob_all,avg_prob_assigned,cluster_csv_path,meta_json_path
0,1,mistralai/Mixtral-8x7B-v0.1,output_proj,randomized,1813382118,8,8,True,5,5,...,eom,0.0,92,23603,32000,0.737594,0.258043,0.983374,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...
1,2,mistralai/Mixtral-8x7B-v0.1,output_proj,randomized,1813382118,158,158,True,5,5,...,eom,0.0,239,27933,32000,0.872906,0.120670,0.949454,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...
2,3,mistralai/Mixtral-8x7B-v0.1,output_proj,randomized,1813382118,1111,1111,True,5,5,...,eom,0.0,962,20414,32000,0.637938,0.342022,0.944648,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...
3,4,mistralai/Mixtral-8x7B-v0.1,output_proj,randomized,1813382118,2156,2156,True,5,5,...,eom,0.0,1079,19386,32000,0.605812,0.374985,0.951285,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...
4,5,mistralai/Mixtral-8x7B-v0.1,output_proj,randomized,1813382118,3052,3052,True,5,5,...,eom,0.0,1093,19269,32000,0.602156,0.379665,0.954307,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...,comp/mistralai/Mixtral-8x7B-v0.1/output_proj/r...


# gpt-oss

In [10]:
model_name = "gpt-oss"
space_name = "output_proj"
# # ===== Step 1: Load tokenizer =====
# tokenizer_path = f"{model_name}/tokenizer" 
# tokenizer = AutoTokenizer.from_pretrained(tokenizer_path)
# print(f"[✓] Tokenizer loaded from: {tokenizer_path}")

# ===== Step 2: Load lm_head.weight =====
space_name = "output_proj"
lm_head_path = f"{model_name}/tensors/{space_name}.pt" 
embedding_matrix = torch.load(lm_head_path, map_location="cpu")
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")
embedding_matrix = embedding_matrix.to(torch.float32)
# embedding_matrix = embedding_matrix.cpu().to(torch.float32).numpy()
print(f"[✓] lm_head.weight loaded: shape = {embedding_matrix.shape}")

[✓] lm_head.weight loaded: shape = torch.Size([201088, 2880])
[✓] lm_head.weight loaded: shape = torch.Size([201088, 2880])


In [11]:
# =========================
# Config
# =========================

out_root = "comp"

full_pca_dim = 2880

candidate_dims = [6, 182, 466, 739, 1591, 2264, 2532, 2868, 2880]

hdbscan_param_grid = [
    {
        "min_cluster_size": 5,
        "min_samples": 5,
        "metric": "euclidean",
        "cluster_selection_method": "eom",
        "cluster_selection_epsilon": 0.0,
    },
    # if needed
    # {
    #     "min_cluster_size": 20,
    #     "min_samples": 10,
    #     "metric": "euclidean",
    #     "cluster_selection_method": "eom",
    #     "cluster_selection_epsilon": 0.0,
    # },
]

l2_norm = True
summary_filename = f"summary.csv"

In [12]:
all_summary = []

for seed in seed_list:
    df = run_random_pca_hdbscan_grid(
        embedding_matrix=embedding_matrix,
        candidate_dims=candidate_dims,
        hdbscan_param_grid=hdbscan_param_grid,
        model_name=model_name,
        space_name=space_name,
        pca_seed=seed,
        randomized_fit_dim=full_pca_dim,
        out_root="comp",
        l2_norm=True,
    )
    all_summary.append(df)

all_summary_df = pd.concat(all_summary, ignore_index=True)
all_summary_df.head()

[warn] randomized_fit_dim is ignored in the corrected per-dim randomized PCA pipeline.
[✓] Randomized PCA finished | model=gpt-oss dim=6 seed=1813382118 time=3.31s cum_var=0.035669
[✓] Saved to: comp/gpt-oss/random_pca/seed_1813382118/pca_dim_6
Memory Cleaned.
[✓] Randomized PCA finished | model=gpt-oss dim=182 seed=1813382118 time=6.58s cum_var=0.233430
[✓] Saved to: comp/gpt-oss/random_pca/seed_1813382118/pca_dim_182
Memory Cleaned.
[✓] Randomized PCA finished | model=gpt-oss dim=466 seed=1813382118 time=10.49s cum_var=0.371812
[✓] Saved to: comp/gpt-oss/random_pca/seed_1813382118/pca_dim_466
Memory Cleaned.
[✓] Randomized PCA finished | model=gpt-oss dim=739 seed=1813382118 time=12.26s cum_var=0.476301
[✓] Saved to: comp/gpt-oss/random_pca/seed_1813382118/pca_dim_739
Memory Cleaned.
[✓] Randomized PCA finished | model=gpt-oss dim=1591 seed=1813382118 time=30.31s cum_var=0.732096
[✓] Saved to: comp/gpt-oss/random_pca/seed_1813382118/pca_dim_1591
Memory Cleaned.
[✓] Randomized PCA fin

,run_id,model_name,space_name,pca_mode,pca_seed,pca_source_dim,pca_dim,l2_norm,min_cluster_size,min_samples,...,cluster_selection_method,cluster_selection_epsilon,n_clusters_excl_noise,n_noise,n_total,noise_ratio,avg_prob_all,avg_prob_assigned,cluster_csv_path,meta_json_path
0,1,gpt-oss,output_proj,randomized,1813382118,6,6,True,5,5,...,eom,0.0,1198,149829,201088,0.745092,0.250480,0.982626,comp/gpt-oss/output_proj/random/seed_181338211...,comp/gpt-oss/output_proj/random/seed_181338211...
1,2,gpt-oss,output_proj,randomized,1813382118,182,182,True,5,5,...,eom,0.0,738,169654,201088,0.843680,0.147158,0.941392,comp/gpt-oss/output_proj/random/seed_181338211...,comp/gpt-oss/output_proj/random/seed_181338211...
2,3,gpt-oss,output_proj,randomized,1813382118,466,466,True,5,5,...,eom,0.0,1918,169215,201088,0.841497,0.153696,0.969675,comp/gpt-oss/output_proj/random/seed_181338211...,comp/gpt-oss/output_proj/random/seed_181338211...
3,4,gpt-oss,output_proj,randomized,1813382118,739,739,True,5,5,...,eom,0.0,2769,155544,201088,0.773512,0.218618,0.965252,comp/gpt-oss/output_proj/random/seed_181338211...,comp/gpt-oss/output_proj/random/seed_181338211...
4,5,gpt-oss,output_proj,randomized,1813382118,1591,1591,True,5,5,...,eom,0.0,3610,136401,201088,0.678315,0.306931,0.954135,comp/gpt-oss/output_proj/random/seed_181338211...,comp/gpt-oss/output_proj/random/seed_181338211...
